In [ ]:
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio
!pip install -q transformers timm peft opencv-python-headless
!pip install -q wandb scikit-learn matplotlib pandas tqdm umap-learn fvcore

In [ ]:
import os
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content/work')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print('cwd:', WORK)

In [ ]:
import zipfile, shutil
ARCHIVE = Path('/kaggle/input/archstyle55/archstyle55.zip')
DATA_ROOT = WORK / 'archstyle55'
if not DATA_ROOT.exists():
    with zipfile.ZipFile(ARCHIVE) as z:
        z.extractall(DATA_ROOT)
list((DATA_ROOT / 'images').iterdir())[:5], list((DATA_ROOT / 'splits').iterdir())[:5]

In [ ]:
REPO = WORK / 'pipeline'
if not REPO.exists():
    !git clone --depth=1 https://github.com/lkrei/archstyle55.git pipeline_repo && cp -r pipeline_repo/pipeline {REPO}
import sys
sys.path.insert(0, str(REPO))
os.environ['ARCH_DATA_DIR'] = str(DATA_ROOT / 'images')
os.environ['ARCH_RESULTS_DIR'] = str(WORK / 'results')

In [ ]:
CONFIG = dict(
    models = ['efficientnet_b1', 'efficientnet_b2', 'efficientnet_b3',
              'efficientnet_v2_s', 'convnext_small', 'vit_b16',
              'dinov2_vitb14_linear'],
    seeds  = [42],
    wandb_project = 'archstyle-vkr',
    use_wandb = False,
)
if CONFIG['use_wandb']:
    import wandb; wandb.login()

In [ ]:
!python -m pipeline.segmentation.run_segmentation --backend segformer

In [ ]:
!python -m pipeline.data.prepare_splits
!python -m pipeline.segmentation.extract_attributes

In [ ]:
for model in CONFIG['models']:
    for seed in CONFIG['seeds']:
        wandb_args = ['--wandb-project', CONFIG['wandb_project']] if CONFIG['use_wandb'] else ['--no-wandb']
        !python -m pipeline.training.run --model {model} --seed {seed} {' '.join(wandb_args)}
        run_dir = Path(os.environ['ARCH_RESULTS_DIR']) / 'runs' / f'{model}_seed{seed}'
        !python -m pipeline.evaluation.evaluate --run-dir {run_dir} --checkpoint last_ema.pt
        !python -m pipeline.reports.make_report --run-dir {run_dir}

In [ ]:
!python -m pipeline.reports.embeddings
!python -m pipeline.reports.compute_cost --models {' '.join(CONFIG['models'])}
!python -m pipeline.scripts.draw_pipeline

In [ ]:
import json
from pipeline.evaluation.clip_zeroshot import zero_shot_predict, save_zero_shot
from pipeline.config import RESULTS_DIR, SPLITS_DIR
splits = json.loads(Path(SPLITS_DIR / 'data_splits.json').read_text())
idx_to_class = json.loads(Path(SPLITS_DIR / 'idx_to_class.json').read_text())
class_names = [idx_to_class[str(i)] for i in range(len(idx_to_class))]
result = zero_shot_predict(splits['test'], class_names, model_id='openai/clip-vit-base-patch16')
save_zero_shot(result, RESULTS_DIR / 'clip_zeroshot')
print('CLIP zero-shot:', result.accuracy, 'top5', result.top5_accuracy)